In [1]:
import sys
sys.path.append("..")

import pandas as pd

short_ic_ranking = pd.read_excel("../exports/RANKING_short_iron_condor.xlsx")

# Filter out gap-risk-flagged AND volatility-persistence-flagged stocks
clean_candidates = short_ic_ranking[
    (short_ic_ranking["gap_risk_flag"].isna() | (short_ic_ranking["gap_risk_flag"] == "")) &
    (short_ic_ranking["volatility_persistence_flag"].isna() | (short_ic_ranking["volatility_persistence_flag"] == ""))
]
top_25 = clean_candidates.head(25)

must_include = short_ic_ranking[short_ic_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[
    (must_include["gap_risk_flag"].isna() | (must_include["gap_risk_flag"] == "")) &
    (must_include["volatility_persistence_flag"].isna() | (must_include["volatility_persistence_flag"] == ""))
]

final_candidates = pd.concat([top_25, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)
final_candidates

,symbol,short_ic_score,gap_risk_flag,Total flips (closed),Median monthly range_pct,recency_weighted_range_pct,Largest monthly range_pct,Median max_adverse_pct,recency_weighted_adverse_pct,Worst max_adverse_pct (4yr),Median duration (days),Worst single gap (4yr),last_month_range_pct,volatility_persistence_flag
0,NIFTY,98.0,NaN,77,5.24,4.86,12.39,1.02,0.91,5.95,15.2,2.32,3.18,NaN
1,SENSEX,97.4,NaN,77,5.50,4.71,12.32,1.06,1.28,5.55,16.0,2.31,3.03,NaN
2,FINNIFTY,97.2,NaN,81,6.10,5.48,15.57,1.27,1.11,7.04,14.2,2.35,3.91,NaN
3,MIDCPNIFTY,96.2,NaN,81,7.01,6.31,16.90,1.86,1.50,7.40,14.0,2.84,2.05,NaN
4,NIFTYNXT50,93.3,NaN,83,7.31,6.80,18.74,1.22,1.06,9.24,15.2,2.80,2.58,NaN
5,BANKNIFTY,90.3,NaN,70,6.10,4.75,17.01,1.52,1.62,7.15,19.6,3.73,2.16,NaN
6,SUNPHARMA,86.8,NaN,53,8.47,7.71,16.84,2.57,2.38,7.31,20.0,3.53,6.04,NaN
7,HDFCBANK,86.2,NaN,63,8.08,8.26,26.94,2.16,1.91,7.56,17.2,3.33,7.00,NaN
8,BAJAJ-AUTO,85.2,NaN,68,10.65,9.36,23.04,2.71,1.90,7.01,15.2,3.22,5.05,NaN
9,DABUR,84.1,NaN,65,8.62,9.08,21.12,2.41,2.41,7.34,19.2,2.66,10.67,NaN


In [2]:
# Incremental refresh — same pattern as 06_daily_screener.ipynb/02_position_monitor.ipynb.
# Previously this did a full fetch(from_date=2022-01-01) for every one of the ~27
# symbols on every run - not destructive (the from_date was always the same, so it
# was never silently shrinking history), just wasteful: re-pulling ~4 years of 4H
# data for a shortlist check that only needs to catch up on what changed since the
# last save. process_stock_incremental() fetches just the recent catch-up window
# and merges onto existing saved history instead.
#
# index_symbols also only had NIFTY/BANKNIFTY before this fix - a real gap, since
# FINNIFTY/SENSEX/MIDCPNIFTY/NIFTYNXT50 are 4 of the top 6 short_ic_score names
# (indices score best here: low range, tight adverse) and were silently skipped
# with "no token found" every run. Added their correct tradingsymbol/exchange pairs.

from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.scale_pipeline import process_stock_incremental
from datetime import datetime

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {
    "NIFTY": ("NIFTY 50", "NSE"),
    "BANKNIFTY": ("NIFTY BANK", "NSE"),
    "FINNIFTY": ("NIFTY FIN SERVICE", "NSE"),
    "SENSEX": ("SENSEX", "BSE"),
    "MIDCPNIFTY": ("NIFTY MID SELECT", "NSE"),
    "NIFTYNXT50": ("NIFTY NEXT 50", "NSE"),
}

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        process_stock_incremental(symbol, token, kite, to_date=datetime.now())
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2026-05-04 to 2026-09-01...
Saved 2302 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/NIFTY_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/NIFTY_full_report.xlsx
[1/25] NIFTY — refreshed
Fetching 2026-05-04 to 2026-09-01...
Saved 2303 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/SENSEX_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/SENSEX_full_report.xlsx
[2/25] SENSEX — refreshed
Fetching 2026-05-04 to 2026-09-01...
Saved 2302 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/FINNIFTY_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/FINNIFTY_full_report.xlsx
[3/25] FINNIFTY — refreshed
Fetching 2026-05-04 to 2026-09-01...
Saved 2274 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/MIDCPNIFTY_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/MIDCPNIFTY_full_report.xlsx
[4/25] MIDCPNIFTY — refreshed
Fetching 2026-05-

In [3]:
from src.monthly_range_screener import screen_monthly_range

live_check = screen_monthly_range(symbols_to_refresh)
live_check.sort_values("current_month_range_pct", ascending=False)

,symbol,days_into_month,current_month_range_pct,median_monthly_range,75th_pct_monthly_range,90th_pct_monthly_range,already_past_median,already_past_75th,already_past_90th
24,BOSCHLTD,1,3.23,10.45,13.10,20.03,False,False,False
8,BAJAJ-AUTO,1,3.12,10.65,14.11,16.75,False,False,False
19,EICHERMOT,1,2.41,11.66,14.64,18.00,False,False,False
7,HDFCBANK,1,2.40,8.08,10.14,13.61,False,False,False
6,SUNPHARMA,1,2.13,8.47,11.10,13.38,False,False,False
10,TCS,1,2.00,9.36,11.14,13.88,False,False,False
13,KOTAKBANK,1,1.98,8.72,11.19,13.68,False,False,False
20,COALINDIA,1,1.94,12.17,14.65,16.67,False,False,False
16,COLPAL,1,1.66,9.28,11.96,18.80,False,False,False
21,CONCOR,1,1.60,11.80,17.26,21.11,False,False,False
